In [7]:
import os
import re
import json
from typing import Dict, Any, List, Tuple, Sequence

import pandas as pd
import networkx as nx

# =========================
# Config (edit if needed)
# =========================
DATA_PATH = "./training_data_normalized.csv"
OUT_BASE = "./add_feature_dataset"

GEXF_PATHS = {
    "NOTEARS": "./graph_NOTEARS.gexf",
    "PC": "./graph_PC.gexf",
    "GES": "./graph_GES.gexf",
    "GOLEM": "./graph_GOLEM.gexf",
}

COL_PREFIX = "edge"
STRICT_WEIGHT = True

# --- train/test split (shared for ALL files) ---
TRAIN_RATIO = 0.7
RANDOM_STATE = 42
SHUFFLE_BEFORE_SPLIT = True

# reproducibility
SAVE_SPLIT_INDICES = True  # keep default True unless you really don't want it

# --- NEW: metadata toggle (default OFF) ---
WRITE_METADATA = False

# --- NEW: shorter common filename prefix ---
DATA_PREFIX = "data_with_features"


# =========================
# Helpers
# =========================
def is_int_str(x: str) -> bool:
    return re.fullmatch(r"-?\d+", str(x)) is not None


def ensure_digraph(G) -> nx.DiGraph:
    if isinstance(G, nx.MultiDiGraph):
        return G
    if isinstance(G, nx.DiGraph):
        return G
    return nx.DiGraph(G)


def build_node_to_col_mapping(df: pd.DataFrame, G: nx.DiGraph) -> Dict[Any, str]:
    cols = list(df.columns)
    nodes = list(G.nodes())

    if all(str(n) in df.columns for n in nodes):
        return {n: str(n) for n in nodes}

    if all(is_int_str(n) for n in nodes):
        idxs = [int(str(n)) for n in nodes]
        if min(idxs) >= 0 and max(idxs) < len(cols):
            return {n: cols[int(str(n))] for n in nodes}

    normalized_cols = {re.sub(r"\s+", "", c): c for c in cols}
    mapping: Dict[Any, str] = {}
    for n in nodes:
        key = re.sub(r"\s+", "", str(n))
        if key not in normalized_cols:
            mapping = {}
            break
        mapping[n] = normalized_cols[key]
    if mapping:
        return mapping

    missing = [str(n) for n in nodes if str(n) not in df.columns]
    raise ValueError(
        "Graph nodes do not match dataset columns, and auto-mapping failed.\n"
        f"- Example missing nodes: {missing[:10]}\n"
        f"- Dataset columns (first 20): {cols[:20]}\n"
        "Fix options:\n"
        "1) Rename GEXF nodes to match CSV column names, or\n"
        "2) Add a manual node->column mapping in code."
    )


def get_edge_weight(attrs: Dict[str, Any]) -> float:
    w = attrs.get("weight", None)
    if w is None:
        w = attrs.get("value", None)
    if w is None:
        raise KeyError("missing weight")
    return float(w)


def make_unique(name: str, existing: set) -> str:
    if name not in existing:
        return name
    i = 1
    while f"{name}__dup{i}" in existing:
        i += 1
    return f"{name}__dup{i}"


def iter_edges_with_attrs(G) -> List[Tuple[Any, Any, Dict[str, Any]]]:
    if isinstance(G, nx.MultiDiGraph):
        out = []
        for u, v, k, attrs in G.edges(keys=True, data=True):
            attrs2 = dict(attrs)
            attrs2["_multikey"] = str(k)
            out.append((u, v, attrs2))
        return out
    return list(G.edges(data=True))


# =========================
# shared split indices
# =========================
def make_shared_split_indices(
    n_rows: int,
    train_ratio: float,
    random_state: int,
    shuffle: bool,
) -> Tuple[List[int], List[int]]:
    if not (0.0 < train_ratio < 1.0):
        raise ValueError(f"train_ratio must be in (0,1). got {train_ratio}")
    if n_rows <= 1:
        raise ValueError(f"Need at least 2 rows to split. n_rows={n_rows}")

    idx = list(range(n_rows))
    if shuffle:
        idx = pd.Series(idx).sample(frac=1.0, random_state=random_state).tolist()

    n_train = int(n_rows * train_ratio)
    if n_train <= 0:
        n_train = 1
    if n_train >= n_rows:
        n_train = n_rows - 1

    train_idx = idx[:n_train]
    test_idx = idx[n_train:]
    return train_idx, test_idx


def split_and_save_by_indices(
    df: pd.DataFrame,
    out_dir: str,
    base_filename_no_ext: str,
    train_idx: Sequence[int],
    test_idx: Sequence[int],
) -> Dict[str, Any]:
    os.makedirs(out_dir, exist_ok=True)

    df_train = df.iloc[list(train_idx)].copy()
    df_test = df.iloc[list(test_idx)].copy()

    out_train = os.path.join(out_dir, f"{base_filename_no_ext}_train.csv")
    out_test = os.path.join(out_dir, f"{base_filename_no_ext}_test.csv")

    df_train.to_csv(out_train, index=False)
    df_test.to_csv(out_test, index=False)

    print(f"[SPLIT] {base_filename_no_ext}: train={len(df_train)}, test={len(df_test)}")
    return {
        "n_rows_total": int(len(df)),
        "n_train": int(len(df_train)),
        "n_test": int(len(df_test)),
        "train_csv": out_train,
        "test_csv": out_test,
    }


def add_edge_features_for_alg(
    df_base: pd.DataFrame,
    alg: str,
    gexf_path: str,
    out_dir: str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    df = df_base.copy()
    original_cols = list(df.columns)

    G = ensure_digraph(nx.read_gexf(gexf_path))
    node_to_col = build_node_to_col_mapping(df, G)

    existing_cols = set(df.columns)
    edges_meta: List[Dict[str, Any]] = []
    added_cols: List[str] = []

    for u, v, attrs in iter_edges_with_attrs(G):
        try:
            w = get_edge_weight(attrs)
        except Exception:
            if STRICT_WEIGHT:
                raise ValueError(f"[{alg}] Edge ({u}->{v}) has no numeric weight.")
            else:
                continue

        u_col = node_to_col[u]
        v_col = node_to_col[v]

        base_name = f"{COL_PREFIX}_{alg}__{str(u)}__{str(v)}"
        if "_multikey" in attrs:
            base_name += f"__k{attrs['_multikey']}"

        new_col = make_unique(base_name, existing_cols)
        df[new_col] = w * df[u_col] * df[v_col]

        existing_cols.add(new_col)
        added_cols.append(new_col)
        edges_meta.append(
            {
                "alg": alg,
                "u": str(u),
                "v": str(v),
                "u_col": u_col,
                "v_col": v_col,
                "weight": w,
                "new_col": new_col,
            }
        )

    os.makedirs(out_dir, exist_ok=True)

    # shorter filename
    out_csv = os.path.join(out_dir, f"{DATA_PREFIX}_{alg}.csv")
    df.to_csv(out_csv, index=False)

    meta = {
        "alg": alg,
        "data_path": DATA_PATH,
        "gexf_path": gexf_path,
        "output_csv": out_csv,
        "n_rows": int(df.shape[0]),
        "n_original_cols": int(len(original_cols)),
        "n_added_edge_features": int(len(added_cols)),
        "added_feature_cols": added_cols,
        "node_to_col_mapping_preview": {str(k): v for k, v in list(node_to_col.items())[:30]},
        "edges": edges_meta,
    }

    # metadata is optional now
    if WRITE_METADATA:
        out_meta = os.path.join(out_dir, f"metadata_{alg}.json")
        with open(out_meta, "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"[DONE:{alg}] added={len(added_cols)} -> {out_csv}")
    return df, meta


def main():
    os.makedirs(OUT_BASE, exist_ok=True)
    df_base = pd.read_csv(DATA_PATH, low_memory=False)

    # 1) shared split indices are created ONCE from df_base row order
    train_idx, test_idx = make_shared_split_indices(
        n_rows=len(df_base),
        train_ratio=TRAIN_RATIO,
        random_state=RANDOM_STATE,
        shuffle=SHUFFLE_BEFORE_SPLIT,
    )

    # 2) save indices (recommended)
    if SAVE_SPLIT_INDICES:
        split_info_path = os.path.join(OUT_BASE, "split_indices.json")
        with open(split_info_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "data_path": DATA_PATH,
                    "train_ratio": TRAIN_RATIO,
                    "random_state": RANDOM_STATE,
                    "shuffle": SHUFFLE_BEFORE_SPLIT,
                    "n_rows": int(len(df_base)),
                    "train_idx": list(map(int, train_idx)),
                    "test_idx": list(map(int, test_idx)),
                },
                f,
                ensure_ascii=False,
                indent=2,
            )
        print(f"[SPLIT] saved indices -> {split_info_path}")

    # 3) split original data and save under ./add_feature_dataset/original/
    original_dir = os.path.join(OUT_BASE, "original")
    split_and_save_by_indices(
        df=df_base,
        out_dir=original_dir,
        base_filename_no_ext=DATA_PREFIX,  # => data_with_features_train/test.csv
        train_idx=train_idx,
        test_idx=test_idx,
    )

    # (optional) original metadata
    if WRITE_METADATA:
        original_meta_path = os.path.join(original_dir, "metadata_original.json")
        with open(original_meta_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "data_path": DATA_PATH,
                    "n_rows": int(len(df_base)),
                    "n_cols": int(df_base.shape[1]),
                    "split_indices_file": os.path.join(OUT_BASE, "split_indices.json")
                    if SAVE_SPLIT_INDICES else None,
                },
                f,
                ensure_ascii=False,
                indent=2,
            )

    # 4) per-ALG: build full csv, then split using the SAME indices
    for alg, gexf_path in GEXF_PATHS.items():
        out_dir = os.path.join(OUT_BASE, alg)
        df_alg, meta_alg = add_edge_features_for_alg(df_base, alg, gexf_path, out_dir)

        split_and_save_by_indices(
            df=df_alg,
            out_dir=out_dir,
            base_filename_no_ext=f"{DATA_PREFIX}_{alg}",
            train_idx=train_idx,
            test_idx=test_idx,
        )

        # (optional) update metadata with split info
        if WRITE_METADATA:
            # re-open and overwrite with split info (simple and consistent)
            split_meta = {
                "train_ratio": TRAIN_RATIO,
                "random_state": RANDOM_STATE,
                "shuffle": SHUFFLE_BEFORE_SPLIT,
                "n_rows_total": int(len(df_alg)),
                "n_train": int(len(train_idx)),
                "n_test": int(len(test_idx)),
            }
            meta_alg["train_test_split"] = split_meta
            out_meta = os.path.join(out_dir, f"metadata_{alg}.json")
            with open(out_meta, "w", encoding="utf-8") as f:
                json.dump(meta_alg, f, ensure_ascii=False, indent=2)


if __name__ == "__main__":
    main()


[SPLIT] saved indices -> ./add_feature_dataset\split_indices.json
[SPLIT] data_with_features: train=12516, test=5365
[DONE:NOTEARS] added=18 -> ./add_feature_dataset\NOTEARS\data_with_features_NOTEARS.csv
[SPLIT] data_with_features_NOTEARS: train=12516, test=5365
[DONE:PC] added=24 -> ./add_feature_dataset\PC\data_with_features_PC.csv
[SPLIT] data_with_features_PC: train=12516, test=5365
[DONE:GES] added=55 -> ./add_feature_dataset\GES\data_with_features_GES.csv
[SPLIT] data_with_features_GES: train=12516, test=5365
[DONE:GOLEM] added=26 -> ./add_feature_dataset\GOLEM\data_with_features_GOLEM.csv
[SPLIT] data_with_features_GOLEM: train=12516, test=5365
